# 09 基于因子分析的学生成绩排序

> **模块**：多元分析 ｜ **数据**：`fa_data.csv`、`score.xlsx` ｜ **依赖**：numpy、scipy、scikit-learn

## 1. 问题背景

因子分析是主成分分析的推广与发展，旨在将具有复杂关系的变量综合为少数几个因子，以再现原始变量与因子之间的相互关系；根据不同的因子，还可以对变量进行分类，属于多元分析中的一种降维处理方法。例如，一个学生的英语、数学、语文成绩都很好，潜在的共性因子可能是智力水平高。因此，因子分析的过程实际上就是寻找共性因子和个性因子并得到最优解释的过程。本实验主要运用主成分法和主因子法进行因子分析，并使用函数建立因子分析模型以解决实际问题。


## 2. 学习目标

- 掌握因子分析的基本原理。
- 掌握因子载荷矩阵的估计方法。
- 掌握基于 `sklearn` 实现因子分析。


## 3. 环境与数据

- Jupyter 开发环境。
- Python 3.10。


## 4. 方法与实现

### 4.1 因子分析基本原理

#### 4.1.1 因子分析与主成分分析的区别

主成分分析和因子分析是两种用于降低变量维数以便于描述、理解和分析的方法。实际上，**主成分分析可以说是因子分析的一个特例**。

- **原理不同**：主成分分析利用降维（线性变换）的思想，在损失很少信息的前提下将多个指标转化为几个不相关的综合指标（主成分），即每个主成分都是原始变量的线性组合，使得主成分比原始变量具有某些更优越的性能，从而简化系统结构，抓住问题实质。因子分析更倾向于从数据出发，描述原始变量的相关关系，由研究原始变量相关矩阵内部的依赖关系出发，将错综复杂的关系的变量表示成少数的公共因子和仅对某一个变量有作用的特殊因子的线性组合而成。
- **线性表示方向不同**：主成分分析中是将主成分表示成各变量的线性组合，而因子分析是将变量表示成各公因子的线性组合。
- **假设条件不同**：主成分分析不需要假设条件；而因子分析需要一些假设。因子分析的假设包括：各个共同因子之间不相关，特殊因子之间也不相关，共同因子和特殊因子之间也不相关。
- **主成分的数量不同**：主成分分析的主成分数量与变量数量一致，只是解释的信息量不同。实际应用时会根据碎石图提取前几个主要的主成分。而因子分析的因子个数需要分析者指定，指定的因子数量不同会导致不同的结果。
- **得到结果唯一性不同**：对于主成分分析，当给定的协方差矩阵或相关矩阵的特征值唯一时，主成分也是唯一的。而因子分析中因子不是唯一的，通过旋转可以得到不同因子。
- **应用范围不同**：在实际的应用过程中，主成分分析常被用作达到目的的中间手段，而非完全的一种分析方法，提取出来的主成分无法清晰地解释其代表的含义。而因子分析是一种完全的分析方法，可确切地得出公共因子。

**因子分析与主成分分析是包含与扩展的关系**：

- **包含关系**：因子分析中提取公因子的方法很多，其中一种就是主成分分析。由此可见，主成分分析只是因子分析的一种方法。
- **扩展关系**：因子分析通过因子轴旋转解决主成分分析解释障碍的方法。因子轴旋转可以使原始变量在公因子（主成分）上的载荷重新分布，从而使原始变量在公因子上的载荷两级分化，这样公因子（主成分）就能够用那些载荷大的原始变量来解释。上述过程就解决了主成分分析的现实含义解释障碍。

#### 4.1.2 因子分析思想

##### 4.1.2.1 数学模型

设 $x = \{x_1, x_2, \ldots, x_p\}^T$ 为一个 $p$ 维随机向量，其均值为 $\mu = \{\mu_1, \mu_2, \ldots, \mu_p\}^T$，协方差矩阵为 $\Sigma = (\sigma_{ij})$。

因子分析的一般模型如下式（$m \leq p$）：

$$
\begin{cases}
x_1 = \mu_1 + a_{11}f_1 + a_{12}f_2 + \ldots + a_{1m}f_m + \epsilon_1 \\
x_2 = \mu_2 + a_{21}f_1 + a_{22}f_2 + \ldots + a_{2m}f_m + \epsilon_2 \\
\vdots \\
x_p = \mu_p + a_{p1}f_1 + a_{p2}f_2 + \ldots + a_{pm}f_m + \epsilon_p \\
\end{cases} \tag{1}

$$

上述模型矩阵表示如下，简记为

$$
x = \mu + Af + \epsilon

$$

其中，$f = \{f_1, f_2, \ldots, f_m\}^T$ 称为公共因子向量，$\epsilon = \{\epsilon_1, \epsilon_2, \ldots, \epsilon_m\}^T$ 称为特殊因子向量，$A = (a_{ij})$ 称为因子载荷矩阵。

##### 4.1.2.2 因子载荷矩阵的统计意义

- **1. $A$ 的元素 $a_{ij}$**

$$
\mathrm{cov}(x_i, f_j) = a_{ij} \quad (i = 1, 2, \ldots, p; j = 1, 2, \ldots, m)

$$

元素 $a_{ij}$ 是 $x_i$ 与 $f_j$ 之间的协方差。如果 $x$ 为各变量已标准化的随机向量，则 $a_{ij}$ 是 $x_i$ 与 $f_j$ 之间的相关系数。此时，元素 $a_{ij}$ 反映了第 $i$ 个变量与第 $j$ 个公共因子的相关重要性，绝对值越大，相关密切程度越高。

- **2. $A$ 的行元素平方和 $h_i^2 = \sum_{j=1}^{m} a_{ij}^2$**

对公式（1）两边求方差，得：

$$
\mathrm{Var}(x_i) = a_{i1}^2 \mathrm{Var}(f_1) + a_{i2}^2 \mathrm{Var}(f_2) + \ldots + a_{im}^2 \mathrm{Var}(f_m) + \mathrm{Var}(\epsilon_i)

$$

设 $\mathrm{Var}(f_m) = 1$，$\mathrm{Var}(\epsilon) = \Lambda = \mathrm{diag}(\sigma_1^2, \sigma_2^2, \ldots, \sigma_p^2)$。

则有：

$$
\mathrm{Var}(x_i) = a_{i1}^2 + a_{i2}^2 + \ldots + a_{im}^2 + \sigma_i^2

$$

其中，$\sigma_{ii} = h_i^2 + \sigma_i^2$。

特征因子 $\epsilon_i$ 的方差 $\sigma_i$ 称为特征方差。当 $x$ 为各变量已标准化的随机向量时，$\sigma_{ii} = 1$，此时有 $h_i^2 + \sigma_i^2 = 1$。公式中，$h_i^2$（变量共同度）反映了公共因子对 $x_i$ 的影响，可以看做是公共因子 $f_i$ 对 $x_i$ 的方差贡献，称为共性方差；$\sigma_i^2$ 是特殊因子 $\epsilon_i$ 对 $x_i$ 的方差贡献，称为特殊方差。如果 $h_i^2$ 的值非常接近 1，$\sigma_i^2$ 非常小，则因子分析的效果好，从原变量空间到公因子空间的转化效果好。如果大部分原有变量的变量共同度均较高（如高于 0.7），则说明所抽取的因子能够反映原有变量的大部分信息（如 70% 以上），仅有较少的信息丢失。也就是说，因子分析的效果较好。因此，变量共同度是衡量因子分析效果的重要指标。

- **3. $A$ 的列元素平方和 $g_j^2 = \sum_{i=1}^{p} a_{ij}^2$**

$g_j^2$ 表示同一因子 $f_j$ 对各变量所提供的方差贡献的总和，反映了因子 $f_j$ 对原有变量总方差的解释能力。因子方差贡献的值越高，说明相应因子的重要性越高。因此，因子的方差贡献和方差贡献率是衡量每一个因子相对重要性的一个尺度。

##### 4.1.2.3 因子分析基本步骤

- **1. 验证数据是否适合进行因子分析（KMO 检验和 Bartlett 球形检验）**。
- **2. 确定因子求解的方法和因子的个数**。

因子载荷矩阵的估计方法包括：

- **2.1 主成分法**。
- **2.2 主因子法**。
- **2.3 最大似然估计法**。
- **3. 因子旋转（正交变换）**。
- **4. 因子得分**。

#### 4.1.3 使用主成分法

设样本方差矩阵 $S$ 的特征值依次为 $\lambda_1 \geq \lambda_2 \geq \ldots \geq \lambda_p \geq 0$，相应的正交单位特征向量为 $t_1, t_2, \ldots, t_p$，则：

$$
S = \hat{A}\hat{A}^T+\hat{D}

$$

$$
\hat{A} = (\sqrt{\lambda_1}t_1,\ldots,\sqrt{\lambda_m}t_m) = (a_{ij})_{p \times m}

$$

$$
\hat{D} = \mathrm{diag}({\hat{\sigma}}_1^2,{\hat{\sigma}}_2^2,\ldots,{\hat{\sigma}}_p^2)

$$

$$
{\hat{\sigma}}_i^2 = s_{ii}-\sum_{j=1}^{m}{\hat{a_{ij}}^2} \quad (i = 1, 2, \ldots, p)

$$

- $m$：因子数，选取的原则是累计贡献率 $(\lambda_1+\lambda_2+\ldots+\lambda_m)/(\lambda_1+\lambda_2+\ldots+\lambda_p)$ 达到一个较高的百分比。
- $\hat{A}$ 和 $\hat{D}$ 就是因子模型的一个解，这个解称为**主成分解**。
- $S-(\hat{A}\hat{A}^T+\hat{D})$ 则成为残差矩阵，其对角线元素为零，当其他非对角线元素都很小时，可以认为取 $m$ 个因子的模型很好地拟合了原始数据。

**表 1. 因子分析与主成分分析的区别**

| 区别                         | 因子分析                                      | 主成分分析                                      |
|------------------------------|-----------------------------------------------|-------------------------------------------------|
| 原理                         | 从数据出发，描述原始变量的相关关系           | 利用降维的思想，转化为不相关的综合指标         |
| 线性表示方向                 | 变量表示成各公因子的线性组合                 | 主成分表示成各变量的线性组合                   |
| 假设条件                     | 需要一些假设，如各共同因子不相关等           | 不需要假设条件                                  |
| 主成分/因子的数量             | 分析者指定，不同数量导致不同结果             | 有几个变量就有几个主成分（信息量不等）           |
| 结果唯一性                   | 因子不是唯一的，通过旋转可以得到不同因子       | 特征值唯一时，主成分也是唯一的                  |
| 应用范围                     | 完全的分析方法，可确切地得出公共因子         | 达到目的的中间手段，提取出来的主成分含义不清      |

**表注：** 本表对比了因子分析与主成分分析的基本区别，强调了因子分析在解释变量间复杂关系上的优势。


In [1]:
import numpy as np, pandas as pd, os
data = pd.read_csv(os.path.join('..','data','fa_data.csv')) #index_col =0))
data.head()

,x1,x2,x3,x4,x5,x6,x7,x8
0,9.963335,19.919823,44.489034,1.779277,3.682794,13.987882,29.026813,134.067081
1,10.227888,20.642880,46.609634,1.768644,3.613432,13.558360,28.624268,136.764298
2,10.257756,20.525205,45.729855,1.742711,3.567432,13.151017,27.294096,123.368502
3,10.356467,20.823008,46.538472,1.859686,3.894248,14.712735,31.082007,143.313294
4,10.472448,20.848223,45.915974,1.760640,3.634424,13.995856,29.425489,140.059137


In [2]:
np.set_printoptions(precision=4, suppress=True, linewidth=120)
cor = np.corrcoef(data.T) # 计算相关矩阵
eig, vec = np.linalg.eig(cor) #计算特征值和特征向量
print('相关矩阵：\n', cor, '\n特征值：', eig, '\n特征向量：\n', vec)

相关矩阵：
 [[1.     0.9226 0.8411 0.756  0.7002 0.6195 0.6325 0.5199]
 [0.9226 1.     0.8507 0.8066 0.775  0.6954 0.6965 0.5962]
 [0.8411 0.8507 1.     0.8702 0.8353 0.7786 0.7872 0.705 ]
 [0.756  0.8066 0.8702 1.     0.918  0.8636 0.869  0.8065]
 [0.7002 0.775  0.8353 0.918  1.     0.9281 0.9347 0.8655]
 [0.6195 0.6954 0.7786 0.8636 0.9281 1.     0.9746 0.9322]
 [0.6325 0.6965 0.7872 0.869  0.9347 0.9746 1.     0.9432]
 [0.5199 0.5962 0.705  0.8065 0.8655 0.9322 0.9432 1.    ]] 
特征值： [6.6221 0.8776 0.1593 0.124  0.0226 0.0799 0.068  0.0464] 
特征向量：
 [[-0.3176  0.5669 -0.3323 -0.1276  0.1055  0.2626  0.5937  0.1362]
 [-0.337   0.4616 -0.3607  0.2591 -0.0961 -0.154  -0.6561 -0.1126]
 [-0.3556  0.2483  0.5605 -0.6523 -0.0001 -0.2183 -0.1566 -0.0029]
 [-0.3687  0.0124  0.5325  0.48   -0.0382  0.5401  0.0147 -0.238 ]
 [-0.3728 -0.1398  0.1534  0.4045  0.1393 -0.4877  0.1578  0.61  ]
 [-0.3644 -0.312  -0.1898 -0.0296  0.5467 -0.254   0.1413 -0.5913]
 [-0.3668 -0.3069 -0.1818 -0.0801 -0.7968 -0.1

In [3]:
# 确定公共因子的个数m,这里使用前m个特征值的比重大于90%的标准
for m in range(1,len(eig)):
    if eig[:m].sum()/eig.sum() >= 0.9:
        print('特征值的比重大于90%时，','m={}'.format(m))
        break

特征值的比重大于90%时， m=2


In [4]:
# 计算因子载荷阵A
a1 = np.sqrt(eig[0])*vec[:,0]
a2 = np.sqrt(eig[1])*vec[:,1]
# 拼接向量
A_pcm = np.vstack((a1,a2)).T    
A_pcm = np.mat(A_pcm)
print('主成分法计算得因子载荷矩阵： \n', A_pcm)

主成分法计算得因子载荷矩阵： 
 [[-0.8172  0.5311]
 [-0.8672  0.4325]
 [-0.9152  0.2326]
 [-0.9488  0.0116]
 [-0.9594 -0.131 ]
 [-0.9377 -0.2923]
 [-0.9438 -0.2875]
 [-0.8799 -0.4112]]


In [5]:
# 1.计算约相关矩阵R*
# 设置特殊方差σ^2的初始估计
# σ^2的初始值为R-1矩阵对角线元素rii的倒数（1-rii）
# np.diag(array)素
# array是一个1维数组时，结果形成一个以一维数组为对角线元素的矩阵
# array是一个二维矩阵时，结果输出矩阵的对角线元素
D_estimated = np.diag(1/np.diag((np.mat(cor).I)))
# 计算约相关矩阵R*
R_yue = cor - D_estimated
R_yue

array([[0.8773, 0.9226, 0.8411, 0.756 , 0.7002, 0.6195, 0.6325, 0.5199],
       [0.9226, 0.888 , 0.8507, 0.8066, 0.775 , 0.6954, 0.6965, 0.5962],
       [0.8411, 0.8507, 0.8449, 0.8702, 0.8353, 0.7786, 0.7872, 0.705 ],
       [0.756 , 0.8066, 0.8702, 0.8837, 0.918 , 0.8636, 0.869 , 0.8065],
       [0.7002, 0.775 , 0.8353, 0.918 , 0.9272, 0.9281, 0.9347, 0.8655],
       [0.6195, 0.6954, 0.7786, 0.8636, 0.9281, 0.955 , 0.9746, 0.9322],
       [0.6325, 0.6965, 0.7872, 0.869 , 0.9347, 0.9746, 0.9668, 0.9432],
       [0.5199, 0.5962, 0.705 , 0.8065, 0.8655, 0.9322, 0.9432, 0.9047]])

2. 使用主因子法

主因子法是对主成分法的一种修正方法。其基本公式为：

$
R^* = R - D = AA^T
$

其中，$R$为变量$x$的相关矩阵，$R^*$为$x$的约相关矩阵。$R^*$的对角线元素即为因子载荷平方$h_i^2$，而非对角线元素则完全与$R$相同。

设$\hat{h}_i^2$为$h_i^2$的初始估计值，则初始的共同方差$\hat{σ}_i^2$的估计值为$\hat{σ}_i^2 = 1 - \hat{h}_i^2$。由此，我们可以构造初始的约相关矩阵$\hat{R}^*$如下：

$$
\hat{R}^* = \hat{R} - \hat{D} =
\begin{bmatrix}
\hat{h}_1^2 & r_{12} & \cdots & r_{1p}\\
r_{21} & \hat{h}_2^2 & \cdots & r_{2p} \\
\vdots & \vdots & \ddots & \vdots \\
r_{p1} & r_{p2} & \cdots & \hat{h}_p^2
\end{bmatrix}

$$

表1. 初始约相关矩阵$\hat{R}^*$的构建方式

其中，$\hat{R} = (r_{ij})_{p×p}$，$\hat{D} = $diag$(\hat{σ}_1^2, \hat{σ}_2^2, ..., \hat{σ}_p^2)$。

接着，计算矩阵$\hat{R}^*$的特征值与特征向量。设矩阵$\hat{R}^*$的特征值依次为$λ_1^* \geq λ_2^* \geq ... \geq λ_p^* \geq 0$，相应的正交单位特征向量为$t_1^*, t_2^*, ..., t_p^*$，则：

$
\hat{A} = (\sqrt{λ_1^*}t_1^*, \sqrt{λ_2^*}t_2^*, ..., \sqrt{λ_m^*}t_m^*)
$

$
\hat{D} = $diag$(\hat{σ}_1^2, \hat{σ}_2^2, ..., \hat{σ}_p^2)
$

最后，因子载荷平方$h_i^2$的估计值$\hat{h}_i^2$可以由下式重新计算得到：

$
\hat{σ}_i^2 = 1 - \hat{h}_i^2 = 1 - \sum_{j=1}^{m}{\hat{a}_{ij}^2} \quad (i=1,2,...,p)
$

In [6]:
# 2.计算特征值和特征向量
eig, vec = np.linalg.eig(R_yue)
print('特征值：', eig)
print('特征向量：\n', vec)
# 3.计算因子载荷矩阵
# 计算因子载荷阵A
a1 = np.sqrt(eig[0])*vec[:,0]
a2 = np.sqrt(eig[1])*vec[:,1]
# 拼接向量
A_pcm = np.vstack((a1,a2)).T    
A_pcm = np.mat(A_pcm)
print('主因子法计算得因子载荷矩阵： \n', A_pcm)

特征值： [ 6.5302  0.7778  0.0506 -0.0532  0.0061 -0.0356 -0.0135 -0.0149]
特征向量：
 [[-0.3158  0.5624  0.321   0.6846 -0.0627 -0.0192  0.0702  0.0546]
 [-0.3357  0.4673  0.2521 -0.5944  0.3061  0.1329 -0.0086 -0.3751]
 [-0.3521  0.2449 -0.2333 -0.3335 -0.5157 -0.4374 -0.1325  0.419 ]
 [-0.3673  0.0271 -0.6026  0.1109 -0.1812  0.6582 -0.0751 -0.1314]
 [-0.374  -0.1296 -0.4517  0.17    0.5418 -0.4992  0.2198 -0.1395]
 [-0.3672 -0.3199  0.2466  0.0574  0.3038  0.0864 -0.7135  0.2998]
 [-0.3703 -0.3188  0.2819 -0.1317  0.0278  0.2537  0.6436  0.4302]
 [-0.342  -0.4288  0.2675  0.0705 -0.4658 -0.1909 -0.0104 -0.6076]]
主因子法计算得因子载荷矩阵： 
 [[-0.8069  0.496 ]
 [-0.8579  0.4121]
 [-0.8997  0.216 ]
 [-0.9386  0.0239]
 [-0.9557 -0.1143]
 [-0.9383 -0.2821]
 [-0.9462 -0.2812]
 [-0.874  -0.3781]]


**3.使用最大似然估计法**

直接使用函数FactorAnalysis()默认使用最大似然法求解

sklearn.decomposition.FactorAnalysis(n_components=None,..., rotation=None,...)
- n_components：因子数
- rotation：因子旋转，‘varimax’, ‘quartimax’可选

In [7]:
from sklearn.decomposition import  FactorAnalysis
# 构建模型（提取两个因子）
fa = FactorAnalysis(n_components = 2)  
# 求解最大方差的模型
fa.fit(data.values)   
print("载荷矩阵为：\n",fa.components_.T)
print("特殊方差为：\n",fa.noise_variance_)

载荷矩阵为：
 [[ 0.2508 -0.2115]
 [ 0.4983 -0.3408]
 [ 1.2152 -0.4831]
 [ 0.0569 -0.0098]
 [ 0.1455 -0.0034]
 [ 0.7587  0.1149]
 [ 1.7256  0.2562]
 [ 8.289   2.2708]]
特殊方差为：
 [0.0096 0.03   0.304  0.0005 0.0019 0.0202 0.0598 6.9252]


使用不同的估计方法，得到的因子载荷矩阵是不同的，因为因子载荷矩阵并不是唯一的。

##### 4.1.3.1 因子旋转

建立因子分析模型的目的不仅是找出公因子，更重要的是弄清各公因子的专业意义，以便对实际问题进行分析。然而在很多情况下，因子分析的主成分解、主因子解及极大似然解中的各公因子的典型代表变量并不是很突出，容易使各公因子的专业意义难于解释，从而达不到因子分析的主要目的。

由于因子载荷矩阵是不唯一的，所以应该对因子载荷矩阵进行旋转。目的是使因子载荷矩阵的结构简化，使载荷矩阵每列或行的元素平方值向0和1两极分化。因子旋转主要有以下三种正交旋转方法：
- 方差最大法
- 四次方最大法
- 等量最大法

对于以上得到的因子载荷提取的公因子不能做出合理解释的问题，接下来进行因子旋转。

factor_analyzer.FactorAnalyzer(rotation=None, n_factors=n, method='principal')
- rotation: 旋转的方式 (包括 None: 不旋转, 'varimax': 最大方差法,'promax': 最优斜交旋转 )
- n_factors: 公因子的数量
- method: 因子分析的方法 ( 包括 'minres' : 最小残差因子法,'principal' : 主成分分析法 )

In [10]:
from factor_analyzer import FactorAnalyzer
# n_factors表示提取的因子数量；method='principal'表示使用主成分法
# rotation='varimax'表示使用方差最大旋转法对因子载荷矩阵进行旋转
fa = FactorAnalyzer(n_factors= 2,method='principal',rotation='varimax')
# 训练数据
fa.fit(data.values)
# 因子旋转后因子载荷矩阵
a=fa.loadings_
print('旋转后的主成分法求得的因子载荷矩阵：\n',a)

旋转后的主成分法求得的因子载荷矩阵：
 [[0.2769 0.9344]
 [0.379  0.8918]
 [0.5452 0.771 ]
 [0.7142 0.6247]
 [0.8148 0.5231]
 [0.903  0.3863]
 [0.9046 0.394 ]
 [0.9363 0.2583]]


c:\Users\yohoten\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


从以上结果可以明显看出前三个变量在因子$f_2$上有相对因子的大载荷，后五个变量在因子$f_1$上有相对因子的大载荷，因子$f_1$有理由称为耐力因子，$f_2$为速度因子。

### 4.2 学生成绩的因子分析模型

某高校数学系为开展研究生的推荐免试工作，对报名参加推荐的52名学生已修过的6门课的考试分数统计如下表（全部数据见数据文件 `../data/score.xlsx`）。这6门课是：数学分析、高等代数、概率论、微分几何、抽象代数和数值分析，其中前3门为基础课采用闭卷考试，后3门为开卷考试。

> *原稿配图未随 notebook 保存，此处暂缺。*

在以往的推荐免试工作中，该系是按照学生6门课成绩的总分进行学业评价，再根据其他要求确定最后的推荐顺序。但是这种排序办法没有考虑到课程之间的相关性，以及开闭卷等因素，丢弃了一些信息。我们的任务是研究在学生评价中如何体现开闭卷的影响，找到成绩背后的潜在因素，并科学地针对考试成绩进行合理排序。

#### 4.2.1 导入数据将其标准化并求相关系数矩阵


In [13]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, warnings
from scipy.stats import zscore
warnings.filterwarnings('ignore')
plt.rc('font',family="SimHei")
plt.rc('axes',unicode_minus=False)
df = pd.read_excel(os.path.join("..","data","score.xlsx"))
df.head()

,学生序号,数学分析,高等代数,概率论,微分几何,抽象代数,数值分析
0,A1,96,85,75,83,91,56
1,A2,82,98,68,95,83,96
2,A3,63,69,78,95,74,79
3,A4,68,71,91,79,99,87
4,A5,71,52,74,63,77,67


In [14]:
df = df.set_index('学生序号')

# 2. 数据标准化
c = zscore(df)

# 3. 求相关系数矩阵
r = np.corrcoef(c.T)  
print('相关系数矩阵为\n', r)

相关系数矩阵为
 [[ 1.      0.1917 -0.1039 -0.1326 -0.1679 -0.0092]
 [ 0.1917  1.     -0.141   0.3    -0.0446  0.2284]
 [-0.1039 -0.141   1.      0.0124  0.0717  0.075 ]
 [-0.1326  0.3     0.0124  1.     -0.0626 -0.135 ]
 [-0.1679 -0.0446  0.0717 -0.0626  1.      0.0131]
 [-0.0092  0.2284  0.075  -0.135   0.0131  1.    ]]


从相关系数矩阵可以发现相关系数均在0.3以上，并且有些变量相关性很强。适合进行因子分析

#### 4.2.2 对样本数据进行 KMO 和巴特利特检验

**充分性检测(Adequacy Test)**

因子分析的前提是具有一定的相关性，因此在做因子分析之前, 我们需要先做充分性检测, 就是数据集中是否能找到这些factor, 我们可以使用下面的方式进行寻找：
- Kaiser-Meyer-Olkin Test（KMO检测）

$\quad$$\quad$KMO值：0.9以上非常好；0.8以上好；0.7一般；0.6差；0.5很差；0.5以下不能接受

- Bartlett’s Test（巴特利特球形检测）：用来检测观察到的变量之间是否关联, 如果检测结果在统计学上不显著, 就不能采用因子分析，巴特利球形检验的值范围在0-1，越接近1，使用因子分析效果越好

In [15]:
from factor_analyzer import factor_analyzer 
# KMO检验
KMO = factor_analyzer.calculate_kmo(data)[1].astype(float)
print('KMO值为：',KMO)
if KMO>=0.7:
    if KMO>0.9:
        print('非常适合因子分析')
    elif KMO>0.8:
        print('很适合因子分析')
    else:
        print('适合因子分析')
else:
    print('不太适合因子分析')
# 巴特利特检验
bar = factor_analyzer.calculate_bartlett_sphericity(data)[1].astype(float)
print('巴特利特球形度检验伴随概率：',bar)
# 设置显著性水平
alpha=0.05
if bar<alpha:
     print('拒绝原假设，适合因子分析')
else:
    print('不拒绝原假设，不太适合因子分析')

KMO值为： 0.9090514972175047
非常适合因子分析
巴特利特球形度检验伴随概率： 5.692303335640772e-60
拒绝原假设，适合因子分析


#### 4.2.3 求相关矩阵的特征值、特征向量，以及各主成分的累计贡献率

In [16]:
val,vec=np.linalg.eig(r)
# 求特征值的累加和
cs=np.cumsum(val)  
print("特征值为：",val,"\n累加和为：",cs)
print("各主成分的贡献率（累加）为：",np.cumsum(val/val.sum()))

特征值为： [0.4454 1.4217 0.8274 0.9519 1.2021 1.1515] 
累加和为： [0.4454 1.867  2.6944 3.6464 4.8485 6.    ]
各主成分的贡献率（累加）为： [0.0742 0.3112 0.4491 0.6077 0.8081 1.    ]


提取前两个公共因子的贡献率已经达到80%以上，所以认为提取两个公共因子是合适的。实际上，根据经验确定提取公因子的个数，是定为特征根中大于1的特征根个数，这与上面得到的结果一致。

#### 4.2.4 构建因子模型求荷载矩阵和特殊方差

In [17]:
from sklearn.decomposition import FactorAnalysis
# 构建模型（提取两个因子）
fa = FactorAnalysis(n_components = 2)  
# 求解最大方差的模型
fa.fit(c)   
print("载荷矩阵为：\n",fa.components_.T)
print("特殊方差为：\n",fa.noise_variance_)

载荷矩阵为：
 [[ 0.0968 -0.3101]
 [ 0.7987 -0.3407]
 [-0.1125  0.1252]
 [ 0.6152  0.5727]
 [-0.0768  0.0017]
 [ 0.1183 -0.3337]]
特殊方差为：
 [0.894  0.2494 0.9716 0.2975 0.9941 0.8743]


根据因子载荷矩阵的输出结果可以得到

$$
\begin{cases}{x^*_1 = 0.9188f_1-0.0980f_2+ε^*_1}\\
{x_2 = 0.8568f_1-0.2400f_2+ε^*_2}\\
{x_3 = 0.8830f_1-0.2583f_2+ε^*_3}\\
{x_4 = -0.4831f_1-0.6582f_2+ε^*_4}\\
{x_5 = -0.6688f_1-0.5597f_2+ε^*_5}\\
{x_6 = -0.6007f_1-0.4323f_2+ε^*_6}\\
\end{cases}

$$
$$Var(ε^*) = {[0.1463,0.2075,0.15478,0.3334,0.2414,0.4513]}^T$$

第一公共因子$f_1$与数学分析、高等代数、概率论三门课程有很强的正相关，说明对这三门课的解释力非常高，所以我们将$f_1$取名为“基础课因子”。而微分几何、抽象代数与数值分析均为开卷考试，$f_2$能解释这3门课，将$f_2$叫作“开闭卷因子”。

#### 4.2.5 因子得分

- 方法一：公式计算。设$X$是标准化变量，公共因子为$f$，$R$为相关系数矩阵，则因子得分$\hat{f} = A^TR^{-1}X$
- 方法二：fit_transform()将原数据转化为得分

In [18]:
# 方法二：
fa = FactorAnalysis(n_components = 2) 
res1 = fa.fit_transform(c)
print('方法二得到的因子得分（部分）\n',res1[:5])
# 方法一：
# 载荷矩阵
A = fa.components_.T
# 相关系数转置
R_inv = np.linalg.inv(r)
# 标准化后数据
X = c.T
res2 = A.T@R_inv@X
res2 = res2.T
print('方法一得到的因子得分（部分）\n',res2[:5])

方法二得到的因子得分（部分）
 [[ 0.8487 -0.2042]
 [ 1.8698 -0.4512]
 [ 0.4447  0.9051]
 [ 0.0712  0.0628]
 [-1.2306  0.1351]]
方法一得到的因子得分（部分）
              0         1
学生序号                    
A1    0.848711 -0.204176
A2    1.869786 -0.451213
A3    0.444697  0.905079
A4    0.071164  0.062796
A5   -1.230645  0.135147


$f_1$和$f_2$两个因子影响程度不同，所以，以两个公共因子$f_1$和$f_2$的方差贡献率所占的比重加权，可以构造一个因子综合得分

$$F = c_1f_1+c_2f_2$$
- $c_1=λ1/(λ1+λ2),c_2=λ2/(λ1+λ2)$

按照因子总得分值的大小对学生进行排序。

In [19]:
# 计算两个因子的权重
w = val[:2]/sum(val[:2]) 
# 计算每个评价对象的因子总分
res3 = np.dot(res1,w) 
df['各科总分'] = df.sum(axis=1)
df['因子1'] = res1[:,0]
df['因子2'] = res1[:,1]
df['因子总分'] = res3
df.head()

,数学分析,高等代数,概率论,微分几何,抽象代数,数值分析,各科总分,因子1,因子2,因子总分
学生序号,,,,,,,,,,
A1,96,85,75,83,91,56,486,0.848711,-0.204176,0.046982
A2,82,98,68,95,83,96,522,1.869786,-0.451213,0.102445
A3,63,69,78,95,74,79,458,0.444697,0.905079,0.795259
A4,68,71,91,79,99,87,495,0.071164,0.062796,0.064793
A5,71,52,74,63,77,67,404,-1.230645,0.135147,-0.190653


In [20]:
# 因子总分从高到低排序
spdf1=df.sort_values(by='因子总分',ascending = False)  
# 各科总分从高到低排序
spdf2=df.sort_values(by='各科总分',ascending=False)  
print("排序结果为：\n",spdf1.head(),'\n',spdf2.head())

排序结果为：
       数学分析  高等代数  概率论  微分几何  抽象代数  数值分析  各科总分       因子1       因子2      因子总分
学生序号                                                                       
A30     50    51   97    99    86    52   435 -0.412828  2.146080  1.535671
A43     50    64   92    98    97    65   466  0.190419  1.504034  1.190681
A40     50    68   70    97    81    62   428  0.404951  1.272832  1.065805
A34     81    78   67    98    64    59   447  0.963428  0.750160  0.801033
A3      63    69   78    95    74    79   458  0.444697  0.905079  0.795259 
       数学分析  高等代数  概率论  微分几何  抽象代数  数值分析  各科总分       因子1       因子2      因子总分
学生序号                                                                       
A44     88    78   95    96    83    97   537  0.922166  0.322756  0.465741
A2      82    98   68    95    83    96   522  1.869786 -0.451213  0.102445
A29     78    98   79    98    77    77   507  1.892986 -0.055726  0.409125
A37     79    79   91    99    57    98   503  1.079781  0.463771  0.610716
A5

In [21]:
# 将因子排序结果保存到 output/score_rank.xlsx（运行产物，不入库）
from pathlib import Path
out_dir = Path('../output')
out_dir.mkdir(parents=True, exist_ok=True)
with pd.ExcelWriter(out_dir / 'score_rank.xlsx', engine='openpyxl') as f:
    pd.DataFrame(spdf1).to_excel(f)

### 4.3 主成分法提取因子

> 来源：教材参考实现《数学建模算法与应用》10.3 节的练习稿（MIT，Zhanghan Wang），代码保持原样，仅按需改写数据路径并清空输出。

#### 因子分析模型

$$X=\mu+\Lambda F+\epsilon$$
其中$E(F)=0,\quad E(\epsilon)=0,\quad Cov(F)=I_m,\quad D(\epsilon)=Cov(\epsilon)=diag(\sigma_1^2,\cdots,\sigma_m^2),\quad Cov(F, \epsilon)=0$

- 原始变量$X$的协方差矩阵分解: $$Cov(X)=\Lambda\Lambda^T+diag(\sigma_1^2,\cdots,\sigma_m^2)$$

- 载荷因子$\alpha_{ij}$反映第$i$个变量和第$j$个公共因子的相关系数. 绝对值越大相关的密切程度越高.

- 变量$X_i$的共同度记为$h_i^2=\sum\limits_{j=1}^m\alpha_{ij}^2$, 又有$1=h_i^2+\sigma_i^2$, 故$h_i^2$越接近1, 因子分析效果越好

- $\Lambda$中各列平方和$S_j=\sum\limits_{i=1}^p\alpha_{ij}^2$, 用于衡量$F_j$的相对重要性.

#### 主成分分析法估计载荷因子

设相关系数矩阵$R$的特征值和对应特征向量分别为: $\lambda_1\ge\lambda_2\ge\cdots\ge\lambda_p$和$\eta_1,\eta_2,\cdots,\eta_p$  
设m<p, 则因子载荷矩阵$\Lambda=[\sqrt{\lambda_1}\eta_1,\sqrt{\lambda_2}\eta_2,\cdots,\sqrt{\lambda_m}\eta_m]$  
特殊因子的方差用$R-\Lambda\Lambda^T$的对角元来估计. 即$\sigma_i^2=1-\sum\limits_{j=1}^m\alpha_{ij}^2$  

#### 因子载荷矩阵的估计方法
1. 主成分分析法(242页);
通过因子旋转来直观的判断因子的实际意义
因子得分: 反过来把公共因子表示为原变量的线性组合.
因子得分函数: $F_j=c_j+\beta_{j1}X_1+\cdots+\beta_{jp}X_p,\ j=1,2,\cdots,m$

    - 因子得分:
        - 巴特莱特因子得分估计: $\hat{F}=(\Lambda^TD^{-1}\Lambda)^{-1}\Lambda^TD^{-1}(X-\mu)$
        - 回归方法因子得分估计: $\hat{F}=(\hat{F}_{ij})_{n\times m}=X_0R^{-1}\Lambda$

In [22]:
import numpy as np
from sklearn.decomposition import PCA
import scipy
import sympy
import pandas as pd

R = np.array([[1.000, 0.577, 0.509, 0.387, 0.462],
              [0.577, 1.000, 0.599, 0.389, 0.322],
              [0.509, 0.599, 1.000, 0.436, 0.426],
              [0.387, 0.389, 0.436, 1.000, 0.523],
              [0.462, 0.322, 0.426, 0.523, 1.000]
             ])

# 列为单位特征向量. 由于R是对称阵, 因此都是正交的特征向量(下面一行可以验证这一点).
# print(np.array([[np.round(np.sum(eigvec[:,i]*eigvec[:,j])) for i in range(R.shape[0])] for j in range(R.shape[1])]))
eigval, eigvec = np.linalg.eig(R)
order = eigval.argsort()[::-1]
eigvec = np.array([eigvec[:, order[i]] for i in range(order.shape[0])]).T
eigval = np.sort(eigval)[::-1]
eigvec = eigvec*np.sign(np.sum(eigvec, axis=0))
# 因子载荷矩阵
Lambda = eigvec*np.sqrt(eigval)
print(eigval, Lambda, sep='\n')

# 信息贡献率
b = np.array([eigval[i]/eigval.sum() for i in range(eigval.shape[0])])
print(b)
# 累积贡献率
alpha = np.array([b[:i+1].sum() for i in range(b.shape[0])])
print(alpha)

m = 2
# 特殊因子方差
var_e = [1-Lambda[i, :m].sum() for i in range(Lambda.shape[0])]
print(var_e)

[2.8567 0.8092 0.5397 0.4515 0.3429]
[[ 0.7836 -0.2162 -0.4494  0.2598 -0.2643]
 [ 0.7726 -0.4581  0.1309  0.1387  0.396 ]
 [ 0.7947 -0.2343  0.2461 -0.4451 -0.2343]
 [ 0.7123  0.4729  0.3973  0.3172 -0.1028]
 [ 0.7119  0.5235 -0.3197 -0.257   0.2255]]
[0.5713 0.1618 0.1079 0.0903 0.0686]
[0.5713 0.7332 0.8411 0.9314 1.    ]
[0.43261684809112677, 0.6855427113216132, 0.4396006070185313, -0.1851956198795386, -0.235447913972767]


In [23]:
from sklearn.datasets import load_digits
from sklearn.decomposition import FactorAnalysis
import numpy as np

X, _ = load_digits(return_X_y=True)
fa = FactorAnalysis(n_components=56, random_state=0)
X_transformed = fa.fit_transform(X)
print(X_transformed.shape)
print(fa.components_.shape)
print(fa.noise_variance_.shape)
print(fa.mean_.shape)
# 变换的公式满足下面这个:
print(np.round(fa.mean_ + np.matmul(X_transformed[0], fa.components_) + fa.noise_variance_, 0))
print(X[0])

(1797, 56)
(56, 64)
(64,)
(64,)
[ 0.  0.  6. 14. 10.  2.  1.  0. -0.  1. 14. 15. 11. 15.  6.  0.  0.  4. 16.  3.  1. 12.  8.  0. -0.  5. 13.  1.  1.
  9.  9. -0.  0.  6.  9.  1.  1. 10.  9.  0.  0.  5. 12.  1.  2. 13.  7.  0.  0.  2. 15.  6. 11. 13.  1.  0. -0.  0.
  7. 14. 11.  1.  0.  1.]
[ 0.  0.  5. 13.  9.  1.  0.  0.  0.  0. 13. 15. 10. 15.  5.  0.  0.  3. 15.  2.  0. 11.  8.  0.  0.  4. 12.  0.  0.
  8.  8.  0.  0.  5.  8.  0.  0.  9.  8.  0.  0.  4. 11.  0.  1. 12.  7.  0.  0.  2. 14.  5. 10. 12.  0.  0.  0.  0.
  6. 13. 10.  0.  0.  0.]


## 5. 小结

本次实验介绍了因子分析的思路和一般步骤，并且通过一个实例使用从factor_analyzer库和sklearn库实现因子分析。

通过本次实验需要掌握以下内容：
- 掌握因子分析基本原理
- 掌握因子载荷矩阵的估计方法
- 掌握基于sklearn实现因子分析